In [ ]:
#

In [2]:
import pandas as pd

comp_df = pd.read_csv("raw_data/ettevotja_rekvisiidid__lihtandmed.csv", delimiter=";")
rep_df = pd.read_csv("raw_data/1.aruannete_yldandmed_kuni_31032026.csv", delimiter=";", low_memory=False)

df = pd.concat([
 pd.read_csv("raw_data/4.2019_aruannete_elemendid_kuni_31032026.csv", delimiter=";"),
 pd.read_csv("raw_data/4.2020_aruannete_elemendid_kuni_31032026.csv", delimiter=";"),
 pd.read_csv("raw_data/4.2021_aruannete_elemendid_kuni_31032026.csv", delimiter=";"),
 pd.read_csv("raw_data/4.2022_aruannete_elemendid_kuni_31032026.csv", delimiter=";"),
 pd.read_csv("raw_data/4.2023_aruannete_elemendid_kuni_31032026.csv", delimiter=";"),
 pd.read_csv("raw_data/4.2024_aruannete_elemendid_kuni_31032026.csv", delimiter=";"),
 pd.read_csv("raw_data/4.2025_aruannete_elemendid_kuni_31032026.csv", delimiter=";"),
 ], ignore_index=True)

# emtak_rev_df = pd.read_csv("data/2.EMTAK_myygitulu_kuni_31032026.csv", delimiter=";")
#
# geo_rev_df = pd.read_csv("data/3.myygitulu_geograafiline_kuni_31032026.csv", delimiter=";")
#
# reports_df = pd.read_csv("data/4.2019_aruannete_elemendid_kuni_31032026.csv", delimiter=";")





In [3]:
df.head()

,report_id,tabel,elemendi_label,elemendi_nimetus,vaartus
0,994318,Bilanss,Käibevarad,CurrentAssets,0.0
1,994318,Bilanss,Omakapital,Equity,946.0
2,994318,Bilanss,Osakapital nimiväärtuses,IssuedCapital,2600.0
3,994318,Bilanss,Põhivarad,NonCurrentAssets,946.0
4,994318,Bilanss,Varad,Assets,946.0


In [5]:
dup_keys = (
    df.groupby(["report_id", "elemendi_nimetus"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
    .sort_values(["count", "report_id"], ascending=[False, True])
)

print(dup_keys.head(20))

         report_id                       elemendi_nimetus  count
56030      1646735                        TotalProfitLoss      4
129837     1653795  SurplusDeficitFromOperatingActivities      4
365554     1675452  SurplusDeficitFromOperatingActivities      4
545972     1691303  SurplusDeficitFromOperatingActivities      4
1227761    1750794  SurplusDeficitFromOperatingActivities      4
1338210    1760792  SurplusDeficitFromOperatingActivities      4
1410672    1767226  SurplusDeficitFromOperatingActivities      4
1540254    1778541                        TotalProfitLoss      4
1646247    1788386  SurplusDeficitFromOperatingActivities      4
1688411    1792562  SurplusDeficitFromOperatingActivities      4
1706586    1794249  SurplusDeficitFromOperatingActivities      4
1760284    1799307                        TotalProfitLoss      4
1822562    1805586  SurplusDeficitFromOperatingActivities      4
1859682    1809426  SurplusDeficitFromOperatingActivities      4
1870145    1810526  Surpl

In [6]:
# Drop the duplicates
df2 = df.dropna(subset=["report_id", "elemendi_nimetus"])
df2 = df2.reset_index(drop=True)
df2 = df2.drop_duplicates(subset=["report_id", "elemendi_nimetus"], keep="last")

In [7]:
# Pivot table
df_wide = df2.pivot(index="report_id", columns="elemendi_nimetus", values="vaartus").reset_index()
df_wide.head()


elemendi_nimetus,report_id,Assets,AssetsConsolidated,AverageNumberOfEmployeesInFullTimeEquivalentUnits,AverageNumberOfEmployeesInFullTimeEquivalentUnitsConsolidated,BusinessIncome,CashAndCashEquivalents,CashAndCashEquivalentsConsolidated,CurrentAssets,CurrentAssetsConsolidated,...,ServiceFeeIncome,ServiceFeeIncomeConsolidated,SurplusDeficitFromOperatingActivities,TotalAnnualPeriodProfitLoss,TotalAnnualPeriodProfitLossConsolidated,TotalProfitLoss,TotalProfitLossBeforeTax,TotalProfitLossBeforeTaxConsolidated,TotalProfitLossConsolidated,TotalRevenue
0,994318,946.0,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,-480.0,NaN,-480.0,-480.0,NaN,NaN,NaN
1,1262635,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN
2,1434010,2370.0,NaN,NaN,NaN,NaN,2370.0,NaN,2370.0,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1434860,2560.0,NaN,0.0,NaN,NaN,2560.0,NaN,2560.0,NaN,...,NaN,NaN,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN
4,1435129,4.0,NaN,0.0,NaN,NaN,NaN,NaN,4.0,NaN,...,NaN,NaN,NaN,-150.0,NaN,0.0,0.0,NaN,NaN,NaN


In [8]:
# Merge data from different datasets
rep_details = rep_df[["report_id", "registrikood", "aruandeaasta", "õiguslik vorm", "staatus", "kas auditeeritud?", "minimaalne kategooria andmete alusel" ]]

df_wide = df_wide.merge(rep_details, on="report_id")


In [9]:
%pip install fastparquet



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /Users/jyria/.pyenv/versions/3.10.18/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
# Salvestame andmed parquet formaadis
df_wide.to_parquet("data/merged_reports.parquet", index=False)